In [12]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic
import json
from IPython.display import Markdown

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:

def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    print("Debug: Prompt for dataset generation: ", prompt)
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    print(text)
    return json.loads(text)
    

In [13]:
dataset = generate_dataset()

type(dataset)
print(dataset)

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)


Debug: Prompt for dataset generation:  
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.


[
    {
        "task": "Write a Python function that extracts the AWS account ID from an ARN string like 'arn:aws:s3:::my-bucket/key'"
    },
    {
        "task": "Create a JSON object that represents an IAM policy allowing read-only access to a specific S3 bucket named 'my-data-bucket'"
    },
    {
        "task": "Write a regular expression that matches valid A

In [3]:
def run_prompt(test_case):
    """Merges the prompt and test case, and runs the prompt through the model."""
    prompt = f"""
Please solve the following task.
Task: {test_case['task']}"""

    messages = []
    
    add_user_message(messages, prompt)
    output = chat(messages)
    return output.strip()

In [4]:
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
You are an expert code reviewer. You need to evaluate this AI-generated solution:

Task: {test_case['task']}
Solution: {output}

Provide your evaluation as structured JSON with the following fields:
- "strength": An array of 1-3 key strengths of the solution.
- "weakness": An array of 1-3 key weaknesses of the solution.
- "reasoning": A brief explanation of your evaluation.
- "score": A score from 1 to 10, where 1 is poor and 10 is excellent.
"""

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_output = chat(messages, stop_sequences=["```"])
    return json.loads(eval_output)

In [5]:
def run_test_case(test_case):
    """Runs a single test case and returns the result."""
    # print(f"Running test case: {test_case['task']}")
    output = run_prompt(test_case)

    # Grade the output using the model
    model_grade = grade_by_model(test_case, output)
    score = model_grade['score']
    reasoning = model_grade['reasoning']
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [6]:
from statistics import mean

def run_eval(dataset):
    """Runs the evaluation on the dataset and returns the results."""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
        
    average_score = mean(result['score'] for result in results)
    print(f"Average score across all test cases: {average_score}")    
    
    return results

In [10]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)
    
results = run_eval(dataset)

Average score across all test cases: 5.666666666666667


In [13]:
# print(json.dumps(results, indent=2))

Markdown(f"### Evaluation Results\n\n{json.dumps(results, indent=2)}")

### Evaluation Results

[
  {
    "output": "# AWS Account ID Extractor from ARN\n\nHere's a Python function that extracts the AWS account ID from an ARN string:\n\n```python\ndef extract_account_id_from_arn(arn: str) -> str:\n    \"\"\"\n    Extracts the AWS account ID from an ARN string.\n    \n    ARN format: arn:partition:service:region:account-id:resource-type/resource-id\n    \n    Args:\n        arn (str): The ARN string to parse\n        \n    Returns:\n        str: The AWS account ID, or empty string if not found\n        \n    Raises:\n        ValueError: If the ARN format is invalid\n    \"\"\"\n    if not arn or not isinstance(arn, str):\n        raise ValueError(\"ARN must be a non-empty string\")\n    \n    parts = arn.split(':')\n    \n    if len(parts) < 6:\n        raise ValueError(f\"Invalid ARN format: {arn}\")\n    \n    # ARN format: arn:partition:service:region:account-id:resource\n    # Index:       0   1           2       3      4           5+\n    account_id = parts[4]\n    \n    # Account ID might be empty for some resources (like S3 buckets)\n    return account_id if account_id else \"\"\n\n\n# Test cases\nif __name__ == \"__main__\":\n    # Test 1: IAM ARN with account ID\n    arn1 = \"arn:aws:iam::123456789012:user/Development/product_1234/*\"\n    print(f\"ARN: {arn1}\")\n    print(f\"Account ID: {extract_account_id_from_arn(arn1)}\")\n    print()\n    \n    # Test 2: S3 bucket ARN (no account ID)\n    arn2 = \"arn:aws:s3:::my-bucket/key\"\n    print(f\"ARN: {arn2}\")\n    print(f\"Account ID: {extract_account_id_from_arn(arn2)}\")\n    print()\n    \n    # Test 3: RDS ARN with account ID\n    arn3 = \"arn:aws:rds:us-east-1:123456789012:db:mydbinstance\"\n    print(f\"ARN: {arn3}\")\n    print(f\"Account ID: {extract_account_id_from_arn(arn3)}\")\n    print()\n    \n    # Test 4: Lambda function ARN\n    arn4 = \"arn:aws:lambda:us-west-2:123456789012:function:my-function\"\n    print(f\"ARN: {arn4}\")\n    print(f\"Account ID: {extract_account_id_from_arn(arn4)}\")\n    print()\n    \n    # Test 5: Error handling\n    try:\n        invalid_arn = \"arn:aws:s3\"\n        extract_account_id_from_arn(invalid_arn)\n    except ValueError as e:\n        print(f\"Error: {e}\")\n```\n\n**Output:**\n```\nARN: arn:aws:iam::123456789012:user/Development/product_1234/*\nAccount ID: 123456789012\n\nARN: arn:aws:s3:::my-bucket/key\nAccount ID: \n\nRDS ARN: arn:aws:rds:us-east-1:123456789012:db:mydbinstance\nAccount ID: 123456789012\n\nARN: arn:aws:lambda:us-west-2:123456789012:function:my-function\nAccount ID: 123456789012\n\nError: Invalid ARN format: arn:aws:s3\n```\n\n## Key Points:\n\n1. **ARN Structure**: `arn:partition:service:region:account-id:resource-type/resource-id`\n2. **Account ID Position**: Always at index 4 (5th element) when split by `:`\n3. **Empty Account IDs**: Some services (like S3 buckets) don't have account IDs, so the function returns an",
    "test_case": {
      "task": "Write a Python function that extracts the AWS account ID from an ARN string like 'arn:aws:s3:::my-bucket/key'"
    },
    "score": 7,
    "reasoning": "The solution demonstrates a solid understanding of AWS ARN structure and implements the core functionality correctly with good defensive programming practices. The code successfully handles the primary use case and includes thoughtful test coverage. However, there are edge case considerations and validation gaps: (1) The minimum part count validation may be overly strict for some valid ARN formats, (2) No format validation of the account ID itself (should be 12 numeric digits when present), and (3) The inability to distinguish between missing vs. empty account IDs reduces robustness. These issues are relatively minor for typical use cases but could cause problems in production systems that need strict ARN validation. The documentation is excellent and the code is readable and maintainable."
  },
  {
    "output": "# IAM Policy for Read-Only S3 Access\n\n```json\n{\n  \"Version\": \"2012-10-17\",\n  \"Statement\": [\n    {\n      \"Sid\": \"ListBucketContents\",\n      \"Effect\": \"Allow\",\n      \"Action\": [\n        \"s3:GetObject\",\n        \"s3:ListBucket\"\n      ],\n      \"Resource\": [\n        \"arn:aws:s3:::my-data-bucket\",\n        \"arn:aws:s3:::my-data-bucket/*\"\n      ]\n    }\n  ]\n}\n```\n\n## Explanation\n\n**Key Components:**\n\n| Component | Purpose |\n|-----------|---------|\n| **Version** | Specifies the policy language version (2012-10-17 is current) |\n| **Statement** | Array containing one or more policy rules |\n| **Sid** | Optional identifier for the statement |\n| **Effect** | \"Allow\" or \"Deny\" - grants permission in this case |\n| **Action** | S3 actions permitted:<br>- `s3:ListBucket` - list bucket contents<br>- `s3:GetObject` - read/download objects |\n| **Resource** | ARN paths:<br>- Bucket itself: `arn:aws:s3:::my-data-bucket`<br>- Objects in bucket: `arn:aws:s3:::my-data-bucket/*` |\n\n## Usage\n\nThis policy can be attached to an IAM user, role, or group to grant read-only access to the specified S3 bucket.",
    "test_case": {
      "task": "Create a JSON object that represents an IAM policy allowing read-only access to a specific S3 bucket named 'my-data-bucket'"
    },
    "score": 7,
    "reasoning": "The solution provides a functional and fundamentally correct IAM policy for read-only S3 access. The syntax is valid, the policy language version is current, and the selected actions appropriately restrict permissions to read operations. However, there are minor architectural issues: the Resource configuration for ListBucket is unnecessarily broad, and the solution lacks discussion of common real-world scenarios (versioned buckets) and security best practices (explicit denies, conditions). The explanation is clear and well-structured, making it accessible to users, but could be more comprehensive regarding edge cases and production considerations."
  },
  {
    "output": "# AWS EC2 Instance ID Regular Expression\n\nHere's the regular expression that matches valid AWS EC2 instance IDs:\n\n```regex\n^i-[a-z0-9]{17}$\n```\n\n## Explanation\n\n| Component | Meaning |\n|-----------|---------|\n| `^` | Start of string (anchor) |\n| `i-` | Literal characters: \"i\" followed by hyphen |\n| `[a-z0-9]` | Character class matching lowercase letters (a-z) or digits (0-9) |\n| `{17}` | Exactly 17 characters from the preceding character class |\n| `$` | End of string (anchor) |\n\n## Examples\n\n### Valid matches:\n```\ni-0a1b2c3d4e5f6g7h8\ni-1234567890abcdef0\ni-aaaaaaaaaaaaaaaaa\n```\n\n### Invalid (won't match):\n```\ni-0a1b2c3d4e5f6g7h      (only 16 characters)\ni-0a1b2c3d4e5f6g7h8i    (18 characters)\n0a1b2c3d4e5f6g7h8       (missing \"i-\" prefix)\ni-0A1B2C3D4E5F6G7H8     (uppercase letters not allowed)\n```\n\n## Usage Examples\n\n**Python:**\n```python\nimport re\npattern = r'^i-[a-z0-9]{17}$'\nif re.match(pattern, 'i-0a1b2c3d4e5f6g7h8'):\n    print(\"Valid EC2 instance ID\")\n```\n\n**JavaScript:**\n```javascript\nconst pattern = /^i-[a-z0-9]{17}$/;\nif (pattern.test('i-0a1b2c3d4e5f6g7h8')) {\n    console.log(\"Valid EC2 instance ID\");\n}\n```",
    "test_case": {
      "task": "Write a regular expression that matches valid AWS EC2 instance IDs (format: i- followed by 17 alphanumeric characters)"
    },
    "score": 3,
    "reasoning": "While the solution demonstrates good regex fundamentals (anchors, quantifiers, character classes) and excellent documentation style, it fundamentally misses the actual AWS EC2 ID format specification. AWS EC2 instance IDs use hexadecimal characters (base-16: 0-9 and a-f only), not all alphanumeric characters. The correct regex should be `^i-[a-f0-9]{17}$`. The provided examples containing letters beyond 'f' are factually incorrect and would mislead developers implementing this solution. This is a critical error that renders the solution unsuitable for production use despite its otherwise polished presentation."
  }
]

In [15]:
print("Evaluation results:")
print(json.dumps(results, indent=2))

Evaluation results:
[
  {
    "output": "# AWS Account ID Extractor from ARN\n\nHere's a Python function that extracts the AWS account ID from an ARN string:\n\n```python\ndef extract_account_id_from_arn(arn: str) -> str:\n    \"\"\"\n    Extracts the AWS account ID from an ARN string.\n    \n    ARN format: arn:partition:service:region:account-id:resource-type/resource-id\n    \n    Args:\n        arn (str): The ARN string to parse\n        \n    Returns:\n        str: The AWS account ID, or empty string if not found\n        \n    Raises:\n        ValueError: If the ARN format is invalid\n    \"\"\"\n    if not arn or not isinstance(arn, str):\n        raise ValueError(\"ARN must be a non-empty string\")\n    \n    parts = arn.split(':')\n    \n    if len(parts) < 6:\n        raise ValueError(f\"Invalid ARN format: {arn}\")\n    \n    # ARN format: arn:partition:service:region:account-id:resource\n    # Index:       0   1           2       3      4           5+\n    account_id = part